# Administrative and Governance Dataset - Kalomo Town Council

**Owner:** Nicholas | **CSC4792 Mini Project, Group Project - Kalomo Town Council, Zambia**

This notebook explains how I built the administrative and governance dataset for Kalomo Town Council. The final file records council leadership, departments, ward councillors, grassroots committees, constituencies, chiefdoms, selected resolutions, and one council report.

The work is supported by two scripts:
- `scripts/scraping/scrape_administrative_governance.py`
- `scripts/cleaning/clean_administrative_governance.py`

The notebook can be run offline after cloning the repository. It checks the collected source notes and the processed CSV instead of scraping the council website every time.


## Step 1 - Finding the relevant council pages

I first browsed the Kalomo Town Council website manually to understand where governance information was located. Unlike the development-plans dataset, which mostly came from downloadable PDFs, this information was spread across ordinary pages and news posts.

The most useful pages were:

- **About Us** (`?page_id=118`) - district background and administrative history
- **District profile** (`?page_id=2242`) - wards, chiefdoms, constituencies, population, and area
- **Civic Leaders** (`?page_id=2871`) - ward councillors grouped by constituency
- **Departments** (`?page_id=770`) and **Institutional Management** (`?page_id=3964`) - council offices and units
- **FAQs** (`?page_id=2259`) - short definitions of CDF and WDC
- Council news posts - names of office holders and examples of grassroots governance structures in use

This browsing stage helped me decide which URLs to put in the scraper's `SOURCE_PAGES` list.


## Step 2 - Scraping approach and access limits

The scraper uses the same basic approach as the rest of the project: `requests` fetches each page, `BeautifulSoup` removes page chrome, and the visible text is saved for later cleaning. The script also disables certificate verification because the council site has the same certificate-chain issue encountered by other team members.

In this working environment, direct requests to `kalomocouncil.gov.zm` returned HTTP 403. That meant I could not rely only on an automated live scrape here. To keep the work moving, I compiled `data/raw/administrative_governance/source_pages_extract.txt` from indexed versions of the same pages, readable council news posts, and direct browser captures supplied while checking the live site.

I kept the fallback visible in the repo because it matters for reproducibility: where the page was fully readable, I used the page content; where it was incomplete, I noted the gap instead of filling it with assumptions.


In [1]:
import sys, os
sys.path.append(os.path.join("..", "scripts", "scraping"))
import scrape_administrative_governance as sag

print("Source pages targeted:")
for name, url in sag.SOURCE_PAGES.items():
    print(f" - {name}: {url}")

Source pages targeted:
 - about_us: https://www.kalomocouncil.gov.zm/?page_id=118
 - district_profile: https://www.kalomocouncil.gov.zm/?page_id=2242
 - civic_leaders: https://www.kalomocouncil.gov.zm/?page_id=2871
 - departments: https://www.kalomocouncil.gov.zm/?page_id=770
 - faqs: https://www.kalomocouncil.gov.zm/?page_id=2259
 - news_index: https://www.kalomocouncil.gov.zm/?page_id=187
 - news_cdf_equipment_commissioning: https://www.kalomocouncil.gov.zm/?p=1799
 - news_2026_budget_consultative_meeting: https://www.kalomocouncil.gov.zm/?p=3782
 - news_mis_launch: https://www.kalomocouncil.gov.zm/?p=4672
 - news_cash_for_work_sensitization: https://www.kalomocouncil.gov.zm/?p=5104


## Step 3 - Issues found while collecting the data

**Access from this environment was limited.** The council website was reachable in a browser, but automated requests from this environment failed. The scraper is still included because it documents the intended collection process and should work from a normal unrestricted connection.

**Some pages did not expose all content in indexed text.** The Civic Leaders and Departments pages appear to use tabbed or dynamic sections. Indexed text only showed part of the Civic Leaders page at first, so I checked the live page directly on 2026-09-11 and captured the full 20-ward councillor list. The Institutional Management page was also checked directly and used to confirm the units under the Office of the Council Secretary.

**Council Secretary names differed by source.** Lisa Mpasela appears in earlier council material, while Trophius Kufanga appears in later material and in the uploaded 2024 meeting minutes. Since the sources did not state the exact handover date, both names are kept with notes explaining the context.

In [2]:
raw_dir = os.path.join("..", "data", "raw", "administrative_governance")
notes_path = os.path.join(raw_dir, "source_pages_extract.txt")
with open(notes_path, encoding="utf-8") as f:
    notes = f.read()
print(f"{notes_path}: {len(notes)} chars of compiled source notes")
print()
print(notes[:1200], "...")

../data/raw/administrative_governance/source_pages_extract.txt: 21062 chars of compiled source notes

SOURCE PAGE EXTRACTS - Kalomo Town Council administrative/governance dataset
Collected by: Nicholas
Method: scripts/scraping/scrape_administrative_governance.py targets these
exact pages on https://www.kalomocouncil.gov.zm/. The council site blocks
automated/agent HTTP clients (returns 403 to a plain requests.get from this
environment, and is robots-disallowed for hosted fetch tools), so the
running notes below were compiled by cross-referencing indexed copies of
each page's rendered text (the same pages the script targets) together with
the council's own news posts, which are separately indexed and were fully
readable. This mirrors the approach already used for the scanned/rotated
PDFs in data/raw/development_plans (documented from listing/context rather
than full text) - content is recorded honestly per source, with gaps noted
rather than invented. Running the script from a normal re

## Step 4 - Turning the notes into rows

After collecting the source notes, I read them manually and entered each confirmed governance fact into `clean_administrative_governance.py`. This was necessary because most of the information was written as prose, not as clean tables.

The final dataset uses these record types:

1. **Leadership** - chairperson, council secretary, directors, council officers, and MPs where they appear in council governance context.
2. **Department** - departments and units directly confirmed by Kalomo council sources.
3. **Ward** - the full list of 20 ward councillors.
4. **Committee** - WDC, CWAC, SDMC, CDFC, headmen, and named WDC chairpersons where available.
5. **Constituency** - Kalomo Central and Dundumwezi.
6. **Chiefdom** - Chikanta, Siachitema, and Sipatunyana.
7. **Resolution** and **Report** - selected governance-level items from meeting minutes.

The cleaner assigns `AG-001` style IDs, standardises missing values as `N/A`, removes duplicate rows, checks that every row has a source, and writes the final CSV using the project-wide pipe delimiter.


## Step 5 - Loading and inspecting the final dataset

In [3]:
import pandas as pd

csv_path = os.path.join("..", "data", "processed", "db-unza26-csc4792-kalomo_town_council_administrative_governance.csv")
df = pd.read_csv(csv_path, sep="|", keep_default_na=False)
df.head(10)

,record_id,record_type,name_or_title,role_or_function,ward,date,source_url,date_scraped
0,AG-001,Contact,Kalomo Town Council - General Contact Information,Email: towncouncilkalomo@gmail.com | Address: ...,N/A,N/A,https://www.kalomocouncil.gov.zm/?page_id=770,2026-09-12
1,AG-002,Leadership,Coy Makaya,Council Chairperson,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
2,AG-003,Leadership,Lisa Mpasela,Council Secretary (per the 2023 IDP launch and...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=1799,2026-09-12
3,AG-004,Leadership,Trophius Kufanga,Council Secretary (per the council's Web-Based...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
4,AG-005,Leadership,Joshua Munsaka Sikaduli,District Commissioner (Office of the President...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
5,AG-006,Leadership,Jimmy Mubanga,Director of Finance,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
6,AG-007,Leadership,Joel Mweempe,Director of Engineering (per the 2026 budget c...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
7,AG-008,Leadership,Judith Beene,Director of Information and Communication Tech...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
8,AG-009,Leadership,Harry Kamboni,"Member of Parliament, Kalomo Central Constitue...",N/A,N/A,https://en.wikipedia.org/wiki/Kalomo_Central,2026-09-12
9,AG-010,Leadership,Valencia Simwale,Vice Council Chairperson; also serves as the N...,Naluja Ward,N/A,https://www.kalomocouncil.gov.zm/?page_id=2871,2026-09-12


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   record_id         88 non-null     str  
 1   record_type       88 non-null     str  
 2   name_or_title     88 non-null     str  
 3   role_or_function  88 non-null     str  
 4   ward              88 non-null     str  
 5   date              88 non-null     str  
 6   source_url        88 non-null     str  
 7   date_scraped      88 non-null     str  
dtypes: str(8)
memory usage: 5.6 KB


In [5]:
print("Rows by record_type:")
print(df["record_type"].value_counts())
print("\nRows with ward = N/A:", (df["ward"] == "N/A").sum(), "out of", len(df))
print("Rows with date = N/A:", (df["date"] == "N/A").sum(), "out of", len(df))

Rows by record_type:
record_type
Leadership      24
Committee       21
Ward            20
Department       9
Resolution       6
Chiefdom         3
Constituency     2
Contact          1
Service          1
Report           1
Name: count, dtype: int64

Rows with ward = N/A: 45 out of 88
Rows with date = N/A: 46 out of 88


## Step 6 - Adding meeting minutes and resolutions

The project brief asks for public council resolutions and reports where available. These were not easy to find on the public website, so I added three scanned council documents that had been supplied directly:

1. Dundumwezi Constituency Development Fund Committee minutes, 29 December 2023
2. Kalomo Central Constituency Development Fund Committee minutes, 9-10 February 2024
3. Community Engagement Meeting on Budget Preparation minutes, 28 November 2024

Because the scans do not have public URLs, rows from these documents cite the document title and date and mark the source as `uploaded scan, no public URL`.

I only extracted the parts that belong in an administrative/governance dataset: committee membership, named WDC chairpersons, council officers, resolutions, and the fact that a budget-performance report was presented. Project tables and detailed financial figures were left for the CDF and financial datasets so the four project files remain clearly separated.

In [6]:
df = pd.read_csv(csv_path, sep="|", keep_default_na=False)
print("Updated record_type breakdown:")
print(df["record_type"].value_counts())
print()
print("Rows citing an uploaded document (no public URL):",
      df["source_url"].str.contains("uploaded scan").sum())
print("Named WDC Chairperson rows:",
      df["name_or_title"].str.contains("WDC\\) Chairperson").sum())

Updated record_type breakdown:
record_type
Leadership      24
Committee       21
Ward            20
Department       9
Resolution       6
Chiefdom         3
Constituency     2
Contact          1
Service          1
Report           1
Name: count, dtype: int64

Rows citing an uploaded document (no public URL): 38
Named WDC Chairperson rows: 14


## Limitations

- The ward councillor list is complete for all 20 wards, but it had to be checked manually from the live Civic Leaders page because indexed text showed only part of the page.
- The department list includes only departments and units that were confirmed from Kalomo-specific sources. I did not add likely departments from other councils.
- Many rows have `date = N/A` because several council pages and indexed news posts did not show a clear publication date.
- The Council Secretary change from Lisa Mpasela to Trophius Kufanga is documented, but the exact handover date was not available - though the uploaded Nov 2024 minutes do explicitly confirm Trophius Kufanga as Council Secretary and meeting chairperson as of that date.
- Named WDC chairpersons are available for 14 of the 20 wards. The dataset still records that WDCs exist across all 20 wards, but it does not invent names for the remaining six.
- Resolution and Report rows are a curated subset of what the uploaded minutes contain, not a full transcription. Detailed CDF project tables and financial performance figures were deliberately left to the team's cdf_projects and financial_data datasets (see Step 6).
- MPs are included only where they help explain the district's civic leadership and constituency structure; they are labelled as national offices, not council staff.

The full column descriptions and coverage notes are in `docs/DATA_DICTIONARY.md`.
